In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

# Statistical Arb in multi pairs trading system based on graph clustering algorithms in US Equities market

https://arxiv.org/pdf/2406.10695


This paper trys to implement stat arb using clustering algrothms, Kelly creiterion, and an ensemble of machine learing classifiers have been used to impove the risk adjusted returns.

In [6]:
import yfinance as yf
import pandas as pd
import numpy as np

# -----------------------------
# STEP 1: DEFINE UNIVERSE
# -----------------------------
tickers = [
# ===== YOUR ORIGINAL =====
"NVDA","MSFT","GOOGL","AMZN","META","AVGO","AMD","TSM","MU",
"PLTR","ORCL","ADBE","ANET","SMCI","DELL","QCOM","ASML",
"AIQ","BOTZ","ARTY","IRBO","THNQ","ROBO","CHAT",

# ===== MEGA / AI CLOUD =====
"AAPL","NFLX","CRM","NOW","SNOW","INTC","IBM","CSCO","UBER","LYFT",
"SHOP","SQ","PYPL","DOCU","WDAY","ZS","CRWD","OKTA","DDOG","NET",

# ===== SEMICONDUCTORS (CORE) =====
"TXN","ADI","NXPI","ON","MCHP","MPWR","SWKS","QRVO","LRCX","KLAC",
"AMAT","TER","ENTG","IPGP","COHR","AEHR","RMBS","LSCC","ALGM","DIOD",

# ===== CHIP DESIGN / AI HARDWARE =====
"ARM","NVTS","INDI","CEVA","GFS","WOLF","SYNA","HIMX","CRUS","SITM",

# ===== DATA / AI INFRA =====
"MDB","ESTC","TEAM","TWLO","PATH","AI","BBAI","SOUN","C3AI","UPST",

# ===== NETWORKING / DATA CENTERS =====
"FTNT","PANW","AKAM","FSLY","CDNS","SNPS","KEYS","NTAP","HPE","DELL",

# ===== ROBOTICS / AUTOMATION =====
"ABB","ROK","FANUY","TER","ISRG","ZBRA","CGNX","OMCL","SYM","KUKAY",

# ===== SEMICONDUCTOR ETFs =====
"SMH","SOXX","SOXQ","PSI","FTXL",

# ===== BROAD TECH / AI ETFs =====
"XLK","VGT","IYW","IGM","IXN",

# ===== MORE AI / THEMATIC ETFs =====
"CHPX","AGIX","JHAI","AIBU",

# ===== OPTIONAL GLOBAL AI / SEMI (ADR/INTL) =====
"005930.KS","000660.KS","2308.TW","2454.TW","2317.TW","6501.T",

# ===== MORE SEMICONDUCTORS =====
"SLAB","POWI","AXTI","FORM","ICHR","ACLS","VECO","CAMT","ONTO","UCTT",
"SMTC","AMKR","ASX","VISL","NPTN","LITE","AAOI","VIAV","FLEX","SANM",

# ===== EDA / DESIGN SOFTWARE =====
"CDNS","SNPS","ANSS","ALTR","MTSI","RMBS","PRFT","IPGP","COHU","KLIC",

# ===== AI / DATA / SOFTWARE =====
"NEWR","BIGC","HUBS","APP","AFRM","BILL","ZI","PD","FIVN","GTLB",
"CFLT","AMPL","DOMO","VRNS","CYBR","TENB","SAIL","QLYS","RPD","WK",

# ===== CYBER + INFRA =====
"ZS","CRWD","OKTA","PANW","FTNT","AKAM","FSLY","NET","DDOG","HPE",

# ===== DATA CENTER / HARDWARE =====
"STX","WDC","PURE","NTNX","VRT","APLD","EQIX","DLR","GDS","SBAC",

# ===== ROBOTICS / AI HARDWARE =====
"ISRG","ZBRA","CGNX","OMCL","SYK","ABBNY","FANUY","KUKAY","YASKY","HUBB",

# ===== MORE GLOBAL SEMIS =====
"8035.T","6857.T","6723.T","7735.T","6920.T","6268.T","6594.T",
"3711.TW","3034.TW","2379.TW","6415.TW","3443.TW",

# ===== ADDITIONAL AI ETFs =====
"WTAI","LOUP","ARKQ","ARKW","KOMP","QTUM","SNSR","XSD","SOXL","SOXS"
]

# -----------------------------
# STEP 1: DOWNLOAD DAILY DATA
# -----------------------------
data = yf.download(
    tickers=tickers,
    start="2000-01-01",
    interval="1d",
    auto_adjust=True,   # important: use adjusted prices
    progress=False
)

# -----------------------------
# STEP 1: EXTRACT CLEAN PRICE PANEL
# -----------------------------
prices = data["Close"].copy()

# if yfinance returns weird multiindex columns, flatten
if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = prices.columns.get_level_values(0)

prices.columns.name = None

# sort dates
prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index()

# drop rows where everything is missing
prices = prices.dropna(how="all")

# forward-fill gaps inside each column
prices = prices.ffill()

# drop columns that still have too little data
min_obs = int(len(prices) * 0.60)   # keep assets with at least 60% coverage
prices = prices.dropna(axis=1, thresh=min_obs)

# optional: clip to first date where at least 8 assets exist
enough_assets_mask = prices.notna().sum(axis=1) >= 8
prices = prices.loc[enough_assets_mask]
prices

$BIGC: possibly delisted; no timezone found
$NEWR: possibly delisted; no timezone found
$ABB: possibly delisted; no timezone found
$SQ: possibly delisted; no timezone found
$KUKAY: possibly delisted; no timezone found
$PRFT: possibly delisted; no timezone found
$NPTN: possibly delisted; no timezone found
$ZI: possibly delisted; no timezone found
$ANSS: possibly delisted; no timezone found
$ALTR: possibly delisted; no timezone found
$C3AI: possibly delisted; no timezone found

11 Failed downloads:
['BIGC', 'NEWR', 'ABB', 'SQ', 'KUKAY', 'PRFT', 'NPTN', 'ZI', 'ANSS', 'ALTR', 'C3AI']: possibly delisted; no timezone found


,000660.KS,005930.KS,2308.TW,2317.TW,2379.TW,2454.TW,3034.TW,3443.TW,3711.TW,6268.T,...,TSM,TXN,UCTT,VECO,VGT,VIAV,WDC,XLK,XSD,ZBRA
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-03,NaN,NaN,NaN,21.776375,NaN,NaN,NaN,NaN,NaN,NaN,...,8.890167,31.313581,NaN,46.984375,NaN,427.758820,2.379010,20.652031,NaN,25.027779
2000-01-04,-2.289120e+05,4283.019531,24.831114,11.539950,22.558973,NaN,NaN,NaN,29.687284,NaN,...,8.948808,29.981890,NaN,45.500000,NaN,389.362915,2.832155,19.604296,NaN,24.666668
2000-01-05,-2.053541e+05,3911.498291,24.300800,11.539950,23.866739,NaN,NaN,NaN,29.426867,NaN,...,9.007455,29.258966,NaN,42.062500,NaN,360.068268,2.643344,19.313263,NaN,25.138889
2000-01-06,-1.946864e+05,3939.535645,25.803354,11.539950,23.049393,NaN,NaN,NaN,29.947714,NaN,...,8.632142,28.459951,NaN,40.000000,NaN,340.728088,2.756631,18.672976,NaN,23.777779
2000-01-07,-1.929084e+05,3883.459229,25.096272,11.074194,23.131119,NaN,NaN,NaN,29.166437,NaN,...,8.960541,28.612139,NaN,37.000000,NaN,409.414093,3.398587,18.998934,NaN,23.513889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-19,1.013000e+06,200500.000000,1455.000000,205.000000,476.500000,1680.0,373.5,2610.0,347.000000,4337.0,...,338.790009,188.289993,62.630001,31.410000,721.719971,34.250000,316.929993,138.429993,331.339996,206.190002
2026-03-20,1.007000e+06,199400.000000,1475.000000,203.000000,479.500000,1700.0,385.0,2645.0,344.000000,4337.0,...,329.239990,187.190002,57.669998,30.650000,705.130005,31.440001,293.100006,135.289993,322.609985,203.619995
2026-03-23,9.330000e+05,186300.000000,1410.000000,196.000000,469.500000,1625.0,372.0,2590.0,329.500000,4065.0,...,338.450012,188.630005,61.119999,31.600000,716.419983,33.610001,294.790009,136.949997,329.339996,205.869995


# Step 2 Returns Matrix

**Simple returns:**
$$
r_t = \frac{P_t}{P_{t-1}} - 1
$$

**Log returns:**
$$
\ell_t = \ln(P_t) - \ln(P_{t-1})
$$

In [11]:
returns = prices.pct_change()

# log returns
log_returns = np.log(prices / prices.shift(1))

# remove the very first row (all NaN by construction)
returns = returns.iloc[1:].copy()
log_returns = log_returns.iloc[1:].copy()


returns

c:\Miniconda3\envs\quant\Lib\site-packages\pandas\core\internals\blocks.py:395: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


,000660.KS,005930.KS,2308.TW,2317.TW,2379.TW,2454.TW,3034.TW,3443.TW,3711.TW,6268.T,...,TSM,TXN,UCTT,VECO,VGT,VIAV,WDC,XLK,XSD,ZBRA
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-04,NaN,NaN,NaN,-0.470070,NaN,NaN,NaN,NaN,NaN,NaN,...,0.006596,-0.042528,NaN,-0.031593,NaN,-0.089761,0.190477,-0.050733,NaN,-0.014428
2000-01-05,-0.102913,-0.086743,-0.021357,0.000000,0.057971,NaN,NaN,NaN,-0.008772,NaN,...,0.006554,-0.024112,NaN,-0.075549,NaN,-0.075237,-0.066667,-0.014845,NaN,0.019144
2000-01-06,-0.051948,0.007168,0.061831,0.000000,-0.034246,NaN,NaN,NaN,0.017700,NaN,...,-0.041667,-0.027308,NaN,-0.049034,NaN,-0.053713,0.042857,-0.033153,NaN,-0.054144
2000-01-07,-0.009133,-0.014234,-0.027403,-0.040360,0.003546,NaN,NaN,NaN,-0.026088,NaN,...,0.038044,0.005347,NaN,-0.075000,NaN,0.201586,0.232877,0.017456,NaN,-0.011098
2000-01-10,-0.009216,0.041516,0.063323,0.021193,0.000000,NaN,NaN,NaN,0.035715,NaN,...,0.040575,0.068484,NaN,0.037162,NaN,0.113581,-0.111112,0.037990,NaN,0.033668
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-19,-0.040720,-0.038369,-0.006826,-0.023810,-0.011411,-0.028902,-0.023529,0.037773,-0.029371,-0.046184,...,-0.002297,-0.013052,0.055088,0.013226,0.002598,0.055795,0.039456,0.003407,0.007817,-0.006361
2026-03-20,-0.005923,-0.005486,0.013746,-0.009756,0.006296,0.011905,0.030790,0.013410,-0.008646,0.000000,...,-0.028189,-0.005842,-0.079195,-0.024196,-0.022987,-0.082044,-0.075190,-0.022683,-0.026348,-0.012464
2026-03-23,-0.073486,-0.065697,-0.044068,-0.034483,-0.020855,-0.044118,-0.033766,-0.020794,-0.042151,-0.062716,...,0.027974,0.007693,0.059823,0.030995,0.016011,0.069020,0.005766,0.012270,0.020861,0.011050


# Step 3 Rebalance

rebalance every 10 days, use 30 day lookback to estimate relationships, use 5 day window for mispricing signals

In [8]:

LOOKBACK = 30      # for correlation / clustering
SIGNAL_WIN = 5     # for mispricing
REBALANCE = 10     # trading frequency

dates = returns.index

# -----------------------------
# STEP 3: BUILD REBALANCE DATES
# -----------------------------

rebalance_idx = np.arange(LOOKBACK, len(dates), REBALANCE)
rebalance_dates = dates[rebalance_idx]

print("Total rebalance points:", len(rebalance_dates))
print("First 5 rebalance dates:", rebalance_dates[:5])
print("Last 5 rebalance dates:", rebalance_dates[-5:])

Total rebalance points: 680
First 5 rebalance dates: DatetimeIndex(['2000-02-15', '2000-02-29', '2000-03-14', '2000-03-28',
               '2000-04-11'],
              dtype='datetime64[ns]', name='Date', freq=None)
Last 5 rebalance dates: DatetimeIndex(['2026-01-19', '2026-02-02', '2026-02-16', '2026-03-02',
               '2026-03-16'],
              dtype='datetime64[ns]', name='Date', freq=None)


# Step 4 Correlation matrix

Look back 30 days take returns of all stocks that exist in that window compute a corr matrix

In [9]:

corr_matrices = {}
valid_assets_per_date = {}

MIN_OBS = 20  # minimum valid days in 30-day window

for date in rebalance_dates:
    
    # get index location
    idx = returns.index.get_loc(date)
    
    # 30-day window
    window = returns.iloc[idx - LOOKBACK: idx]
    
    # count valid observations per asset
    valid_counts = window.notna().sum()
    
    # keep only assets with enough data
    valid_assets = valid_counts[valid_counts >= MIN_OBS].index.tolist()
    
    # subset
    window_clean = window[valid_assets]
    
    # drop remaining rows with NaNs (pairwise clean)
    window_clean = window_clean.dropna()
    
    # skip if too few assets
    if window_clean.shape[1] < 4:
        continue
    
    # compute correlation
    corr = window_clean.corr()
    
    corr_matrices[date] = corr
    valid_assets_per_date[date] = valid_assets

# -----------------------------
# DEBUG OUTPUT
# -----------------------------
print("Total usable correlation matrices:", len(corr_matrices))

sample_date = list(corr_matrices.keys())[0]
print("\nSample date:", sample_date)
print("Assets used:", valid_assets_per_date[sample_date])
print("\nCorrelation matrix:")
print(corr_matrices[sample_date].round(2))

Total usable correlation matrices: 680

Sample date: 2000-02-15 00:00:00
Assets used: ['000660.KS', '005930.KS', '2308.TW', '2317.TW', '2379.TW', '3711.TW', '6501.T', '6857.T', '8035.T', 'AAPL', 'ADBE', 'ADI', 'AEHR', 'AKAM', 'AMAT', 'AMD', 'AMKR', 'AMZN', 'ASML', 'AXTI', 'CDNS', 'CGNX', 'COHR', 'COHU', 'CRUS', 'CSCO', 'DIOD', 'FLEX', 'HUBB', 'IBM', 'INTC', 'KLAC', 'KLIC', 'LRCX', 'LSCC', 'MCHP', 'MSFT', 'MU', 'NTAP', 'NVDA', 'ORCL', 'POWI', 'PURE', 'QCOM', 'RMBS', 'ROK', 'SANM', 'SBAC', 'SMTC', 'SNPS', 'SWKS', 'SYK', 'TER', 'TSM', 'TXN', 'VECO', 'VIAV', 'WDC', 'XLK', 'ZBRA']

Correlation matrix:
           000660.KS  005930.KS  2308.TW  2317.TW  2379.TW  3711.TW  6501.T  \
000660.KS       1.00       0.67     0.17     0.18     0.15     0.27    0.34   
005930.KS       0.67       1.00     0.41     0.20    -0.07     0.29    0.19   
2308.TW         0.17       0.41     1.00     0.60     0.30     0.73   -0.01   
2317.TW         0.18       0.20     0.60     1.00     0.27     0.60   -0.12   
2

# Step 5 graph construction

clustering algos do not use matrixs they use graphs, so lets change correaltion into a weighted graph
We will use a signed graph, the idea of signed graph is to also take direction into consideration not just strength

We split this into 2 matrices.
Given a correlation matrix $C$, define:

$$
A^+ = \max(C, 0)
$$

$$
A^- = \max(-C, 0)
$$

\textbf{Interpretation:}

$A^+$ keeps only positive correlations

$A^-$ keeps only negative correlations (as positive magnitudes)



For each entry $C_{ij}$:

$$
A^+_{ij} =
\begin{cases}
C_{ij}, & C_{ij} > 0 \\
0, & \text{otherwise}
\end{cases}
$$

$$
A^-_{ij} =
\begin{cases}
- C_{ij}, & C_{ij} < 0 \\
0, & \text{otherwise}
\end{cases}
$$



If $C_{ij} = 0.6$:
$$
A^+_{ij} = 0.6, \quad A^-_{ij} = 0
$$

If $C_{ij} = -0.6$:
$$
A^+_{ij} = 0, \quad A^-_{ij} = 0.6
$$



Positive correlation $\rightarrow$ pull-together force

Negative correlation $\rightarrow$ push-apart force



In [12]:

A_plus_dict = {}
A_minus_dict = {}
signed_graph_info = {}

for date, corr in corr_matrices.items():
    C = corr.copy()

    # remove self-correlation first
    np.fill_diagonal(C.values, 0.0)

    # positive and negative parts
    A_plus = C.clip(lower=0)
    A_minus = (-C).clip(lower=0)

    # just to be safe: zero diagonals
    np.fill_diagonal(A_plus.values, 0.0)
    np.fill_diagonal(A_minus.values, 0.0)

    A_plus_dict[date] = A_plus
    A_minus_dict[date] = A_minus

    signed_graph_info[date] = {
        "n_assets": C.shape[0],
        "positive_edge_weight_sum": A_plus.values.sum() / 2,
        "negative_edge_weight_sum": A_minus.values.sum() / 2,
        "num_positive_edges": int((A_plus.values > 0).sum() / 2),
        "num_negative_edges": int((A_minus.values > 0).sum() / 2),
    }

# -----------------------------
# DEBUG
# -----------------------------
sample_date = list(A_plus_dict.keys())[0]

print("Sample date:", sample_date)
print("\nA_plus (positive graph):")
print(A_plus_dict[sample_date].round(2))

print("\nA_minus (negative graph):")
print(A_minus_dict[sample_date].round(2))

print("\nSigned graph summary:")
print(signed_graph_info[sample_date])

Sample date: 2000-02-15 00:00:00

A_plus (positive graph):
           000660.KS  005930.KS  2308.TW  2317.TW  2379.TW  3711.TW  6501.T  \
000660.KS       0.00       0.67     0.17     0.18     0.15     0.27    0.34   
005930.KS       0.67       0.00     0.41     0.20     0.00     0.29    0.19   
2308.TW         0.17       0.41     0.00     0.60     0.30     0.73    0.00   
2317.TW         0.18       0.20     0.60     0.00     0.27     0.60    0.00   
2379.TW         0.15       0.00     0.30     0.27     0.00     0.44    0.00   
3711.TW         0.27       0.29     0.73     0.60     0.44     0.00    0.00   
6501.T          0.34       0.19     0.00     0.00     0.00     0.00    0.00   
6857.T          0.35       0.16     0.10     0.00     0.12     0.00    0.66   
8035.T          0.31       0.12     0.09     0.00     0.05     0.00    0.60   
AAPL            0.01       0.00     0.00     0.00     0.00     0.00    0.03   
ADBE            0.00       0.02     0.00     0.00     0.00     0.02    0

# Step 6 Cluster

A+ moves together
A- moves against

Then SPONGEsym decomposes  the signed graph into positive and negative parts builds normalized matrcies and solves a gernalized eigenvalue probelm then applys k measn to it.

In [14]:
from scipy.linalg import eigh
from sklearn.cluster import KMeans

# -------------------------------------------------
# STEP 6A: choose number of clusters from 90% variance
# -------------------------------------------------

def choose_k_from_corr_90pct(corr: pd.DataFrame, min_k: int = 2) -> int:
    C = corr.values.copy()
    C = 0.5 * (C + C.T)

    eigvals = np.linalg.eigvalsh(C)
    eigvals = np.sort(eigvals)[::-1]
    eigvals = np.clip(eigvals, 0, None)

    total = eigvals.sum()
    if total <= 0:
        return min_k

    cum = np.cumsum(eigvals) / total
    k_90 = int(np.searchsorted(cum, 0.90) + 1)

    # 🔴 IMPORTANT FIX
    n = C.shape[0]
    k_cap = int(np.sqrt(n))

    k = min(k_90, k_cap)
    k = max(min_k, k)

    return k


# -------------------------------------------------
# STEP 6B: helpers for SPONGEsym
# -------------------------------------------------

def diag_inv_sqrt_from_degree(deg: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    """
    Returns diag(1 / sqrt(deg_i)) with safe handling of zeros.
    """
    inv_sqrt = np.zeros_like(deg, dtype=float)
    mask = deg > eps
    inv_sqrt[mask] = 1.0 / np.sqrt(deg[mask])
    return np.diag(inv_sqrt)


def spongesym_cluster(A_plus: pd.DataFrame,
                      A_minus: pd.DataFrame,
                      corr_for_k: pd.DataFrame,
                      random_state: int = 42):
    """
    Paper-style SPONGEsym approximation:
    - choose k via 90% explained variance of correlation matrix
    - build normalized signed operators
    - solve generalized eigenproblem
    - cluster embedding with k-means++
    """

    assets = list(A_plus.index)
    n = len(assets)

    # choose k from correlation matrix
    k = choose_k_from_corr_90pct(corr_for_k)

    Ap = A_plus.values.astype(float)
    Am = A_minus.values.astype(float)

    # symmetry cleanup
    Ap = 0.5 * (Ap + Ap.T)
    Am = 0.5 * (Am + Am.T)

    # degree vectors
    d_plus = Ap.sum(axis=1)
    d_minus = Am.sum(axis=1)

    D_plus = np.diag(d_plus)
    D_minus = np.diag(d_minus)

    # Laplacians
    L_plus = D_plus - Ap
    L_minus = D_minus - Am

    # normalized forms
    Dp_inv_sqrt = diag_inv_sqrt_from_degree(d_plus)
    Dm_inv_sqrt = diag_inv_sqrt_from_degree(d_minus)

    Lp_sym = Dp_inv_sqrt @ L_plus @ Dp_inv_sqrt
    Lm_sym = Dm_inv_sqrt @ L_minus @ Dm_inv_sqrt

    # regularization terms tau^- and tau^+
    # small positive regularization to stabilize generalized problem
    tau_minus = max(np.mean(d_minus), 1e-8)
    tau_plus = max(np.mean(d_plus), 1e-8)

    A_mat = Lp_sym + tau_minus * np.eye(n)
    B_mat = Lm_sym + tau_plus * np.eye(n)

    # numerical symmetry cleanup
    A_mat = 0.5 * (A_mat + A_mat.T)
    B_mat = 0.5 * (B_mat + B_mat.T)

    # generalized eigenproblem:
    # k smallest generalized eigenvectors
    eigvals, eigvecs = eigh(A_mat, B_mat)

    order = np.argsort(eigvals)
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    U = eigvecs[:, :k]

    # row normalization before kmeans++ is standard spectral-clustering practice
    row_norms = np.linalg.norm(U, axis=1, keepdims=True)
    row_norms[row_norms == 0] = 1.0
    Y = U / row_norms

    # k-means++
    km = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=20,
        random_state=random_state
    )
    labels = km.fit_predict(Y)

    label_series = pd.Series(labels, index=assets, name="cluster")

    members = {}
    for c in sorted(label_series.unique()):
        members[int(c)] = label_series[label_series == c].index.tolist()

    debug = {
        "k": k,
        "tau_plus": tau_plus,
        "tau_minus": tau_minus,
        "gen_eigvals_head": eigvals[:10]
    }

    return label_series, members, debug


# -------------------------------------------------
# STEP 6C: run SPONGEsym on every rebalance date
# -------------------------------------------------

cluster_labels_dict = {}
cluster_members_dict = {}
cluster_debug_dict = {}

for date in A_plus_dict.keys():
    A_plus = A_plus_dict[date]
    A_minus = A_minus_dict[date]
    corr = corr_matrices[date]

    # need at least a few names to cluster meaningfully
    if A_plus.shape[0] < 6:
        continue

    labels, members, debug = spongesym_cluster(
        A_plus=A_plus,
        A_minus=A_minus,
        corr_for_k=corr,
        random_state=42
    )

    cluster_labels_dict[date] = labels
    cluster_members_dict[date] = members
    cluster_debug_dict[date] = debug


# -------------------------------------------------
# STEP 6D: inspect one sample date
# -------------------------------------------------

sample_date = list(cluster_labels_dict.keys())[0]

print("Sample date:", sample_date)
print("Chosen k:", cluster_debug_dict[sample_date]["k"])
print("tau_plus:", cluster_debug_dict[sample_date]["tau_plus"])
print("tau_minus:", cluster_debug_dict[sample_date]["tau_minus"])
print("First generalized eigenvalues:")
print(cluster_debug_dict[sample_date]["gen_eigvals_head"])
print()

for c, names in cluster_members_dict[sample_date].items():
    print(f"Cluster {c} ({len(names)} assets):")
    print(names)
    print()

Sample date: 2000-02-15 00:00:00
Chosen k: 7
tau_plus: 11.214536205750486
tau_minus: 2.333703470259645
First generalized eigenvalues:
[0.19632356 0.22751282 0.23542937 0.24304746 0.25064593 0.25159326
 0.25548675 0.25890243 0.26198447 0.26377551]

Cluster 0 (7 assets):
['AEHR', 'CSCO', 'KLIC', 'NTAP', 'ORCL', 'ROK', 'WDC']

Cluster 1 (13 assets):
['ADBE', 'ADI', 'AMZN', 'AXTI', 'DIOD', 'LSCC', 'MCHP', 'MU', 'SANM', 'TSM', 'TXN', 'XLK', 'ZBRA']

Cluster 2 (10 assets):
['AAPL', 'CRUS', 'INTC', 'LRCX', 'MSFT', 'NVDA', 'POWI', 'RMBS', 'SWKS', 'TER']

Cluster 3 (6 assets):
['000660.KS', '005930.KS', '2308.TW', '2317.TW', '2379.TW', '3711.TW']

Cluster 4 (6 assets):
['6501.T', '6857.T', '8035.T', 'AMKR', 'SBAC', 'VECO']

Cluster 5 (7 assets):
['AKAM', 'COHR', 'FLEX', 'HUBB', 'IBM', 'PURE', 'SYK']

Cluster 6 (11 assets):
['AMAT', 'AMD', 'ASML', 'CDNS', 'CGNX', 'COHU', 'KLAC', 'QCOM', 'SMTC', 'SNPS', 'VIAV']



# Step 7 compute cluster relative mispricing

stocks revert relative to their cluster

\textbf{Step-by-step logic}

At each rebalance date $t$:

\textbf{1. Look back 5 days}

$$
R_i^{(5)}(t) = \text{cumulative return over last 5 days}
$$

\textbf{2. Compute cluster average}

For cluster $c$:

$$
\bar{R}_c(t) = \frac{1}{|c|} \sum_{i \in c} R_i^{(5)}(t)
$$

\textbf{3. Compute deviation (THIS IS THE SIGNAL)}

$$
\text{signal}_i = R_i^{(5)}(t) - \bar{R}_c(t)
$$

\textbf{Interpretation:}

Negative $\rightarrow$ underperformed cluster $\rightarrow$ buy

Positive $\rightarrow$ outperformed cluster $\rightarrow$ short

In [15]:
# -----------------------------
# STEP 7: MISPRICING SIGNAL
# -----------------------------

signal_dict = {}

for date in cluster_labels_dict.keys():
    
    labels = cluster_labels_dict[date]
    
    idx = returns.index.get_loc(date)
    
    # 5-day cumulative return
    window = returns.iloc[idx - SIGNAL_WIN: idx]
    
    # cumulative return
    cumret = (1 + window).prod() - 1
    
    # align with cluster assets
    cumret = cumret[labels.index]
    
    signals = pd.Series(index=labels.index, dtype=float)
    
    # compute cluster-relative deviation
    for cluster_id in labels.unique():
        
        members = labels[labels == cluster_id].index
        
        cluster_mean = cumret[members].mean()
        
        signals[members] = cumret[members] - cluster_mean
    
    signal_dict[date] = signals

# -----------------------------
# DEBUG
# -----------------------------
sample_date = list(signal_dict.keys())[0]

print("Sample signals:")
print(signal_dict[sample_date].sort_values())

Sample signals:
MSFT        -0.224205
TER         -0.153681
WDC         -0.150335
AAPL        -0.143212
INTC        -0.140604
LRCX        -0.119886
8035.T      -0.113153
CDNS        -0.105988
ROK         -0.093024
MU          -0.086361
000660.KS   -0.080756
HUBB        -0.078820
SYK         -0.078816
TSM         -0.070446
FLEX        -0.065882
COHR        -0.058329
AKAM        -0.044895
3711.TW     -0.044771
6857.T      -0.041326
2379.TW     -0.036789
SANM        -0.035077
QCOM        -0.034838
IBM         -0.034501
DIOD        -0.025157
6501.T      -0.024574
SBAC        -0.022276
KLIC        -0.020710
ADI         -0.018708
SNPS        -0.011956
ZBRA        -0.009801
CGNX        -0.009691
ASML        -0.009469
AMZN        -0.009401
XLK         -0.008180
ORCL        -0.005653
TXN         -0.000972
CSCO        -0.000756
KLAC         0.004219
CRUS         0.008809
COHU         0.011127
VIAV         0.014015
2308.TW      0.016902
AXTI         0.017642
SMTC         0.022331
LSCC         0.0

In [16]:
sample_date = list(signal_dict.keys())[0]
labels = cluster_labels_dict[sample_date]
signals = signal_dict[sample_date]

cluster_check = pd.DataFrame({
    "signal": signals,
    "cluster": labels
}).groupby("cluster")["signal"].sum()

print(cluster_check)

cluster
0    0.000000e+00
1    5.204170e-18
2    2.775558e-17
3    0.000000e+00
4    0.000000e+00
5   -4.163336e-17
6   -3.469447e-18
Name: signal, dtype: float64


# Turn the signals into raw long/short candidates trades

Stocks below cluster mean = long
Stocks above cluster mean = short

In [17]:
raw_trades_dict = {}

for date, signals in signal_dict.items():
    labels = cluster_labels_dict[date]

    trades = pd.DataFrame({
        "signal": signals,
        "cluster": labels
    })

    trades["side"] = np.where(
        trades["signal"] < 0, "LONG",
        np.where(trades["signal"] > 0, "SHORT", "FLAT")
    )

    # magnitude of deviation can be useful later
    trades["abs_signal"] = trades["signal"].abs()

    raw_trades_dict[date] = trades.sort_values("signal")

# -----------------------------
# DEBUG
# -----------------------------
sample_date = list(raw_trades_dict.keys())[0]

print("Sample raw trades:")
print(raw_trades_dict[sample_date].head(15))
print()
print(raw_trades_dict[sample_date].tail(15))
print()
print(raw_trades_dict[sample_date]["side"].value_counts())

Sample raw trades:
             signal  cluster  side  abs_signal
MSFT      -0.224205        2  LONG    0.224205
TER       -0.153681        2  LONG    0.153681
WDC       -0.150335        0  LONG    0.150335
AAPL      -0.143212        2  LONG    0.143212
INTC      -0.140604        2  LONG    0.140604
LRCX      -0.119886        2  LONG    0.119886
8035.T    -0.113153        4  LONG    0.113153
CDNS      -0.105988        6  LONG    0.105988
ROK       -0.093024        0  LONG    0.093024
MU        -0.086361        1  LONG    0.086361
000660.KS -0.080756        3  LONG    0.080756
HUBB      -0.078820        5  LONG    0.078820
SYK       -0.078816        5  LONG    0.078816
TSM       -0.070446        1  LONG    0.070446
FLEX      -0.065882        5  LONG    0.065882

             signal  cluster   side  abs_signal
POWI       0.033124        2  SHORT    0.033124
005930.KS  0.037266        3  SHORT    0.037266
AMD        0.059368        6  SHORT    0.059368
AMAT       0.060882        6  SHORT 

In [18]:
sample_date = list(raw_trades_dict.keys())[0]
sample_trades = raw_trades_dict[sample_date]

print(sample_trades.groupby("cluster")["side"].value_counts().unstack(fill_value=0))

side     LONG  SHORT
cluster             
0           5      2
1           9      4
2           5      5
3           3      3
4           4      2
5           6      1
6           5      6


# Step 9 Build Labels

Train a classifier to predict which signals will work

A good signal means trades reaches threshold T before next rebalance or not lossing after cost final returns > transaction cost